# CyVerse drone production: Data Store → calibration/cameras → translation/QA → verified upload

Use the matching SpectralBridge checkout **inside the CyVerse VM**, on persistent disk. This notebook runs one flight at a time through the public drone pipeline. It preserves original sources, records blocked/failed flights, and resumes valid stages. It does not perform NEON downloads or a pixel-scale bulk cache.

The incoming brightness JSON supplies per-band calibration gains/offsets in fractional reflectance, separate from the AOP-derived translation registry. No field coefficients are provided here. Camera geometry requires an explicit surface/vertical-datum and camera-assignment policy. A nearest-camera scenario is an assumption, not recovered mosaic contributors. Correction/model QA can still block individual flights.

Run a one-flight pilot and inspect its QA before enabling the complete population. Existing historical notebooks remain unchanged apart from links to the current workflow.


## 1. Install and authenticate in the VM

From a terminal in the repository, install `python -m pip install -e '.[tests,analytics,landsat]'` into the notebook kernel's environment. Install Linux GoCommands and authenticate with `gocmd init` in the terminal. Never put credentials in this notebook. Restart the kernel after updating the package. Transfer/push the matching source and scripts; copying this notebook alone is insufficient.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

REPO = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / 'src/spectralbridge').is_dir() and (path / 'pyproject.toml').is_file())
WORK = Path.home() / 'data/spectralbridge_drone'  # Set to your persistent VM disk.
RUN_ID = 'drone_2023_2024_calibrated_v1'
YEARS = (2023, 2024)
REMOTE_PARENT = 'i:/iplant/home/shared/earthlab/macrosystems/field-data/output'
RASTER_SOURCES = {year: f'{REMOTE_PARENT}/summer-{year}-10cm-10k' for year in YEARS}
CAMERA_SOURCES = {year: f'{REMOTE_PARENT}/summer-{year}' for year in YEARS}
REMOTE_RESULTS_PARENT = f'{REMOTE_PARENT}/spectralbridge_processed'
MANIFEST = REPO / 'src/spectralbridge/data/drone_field_manifest.csv'
POLYGONS = REPO / 'Datasets/niwot_aop_polygons_2023_12_8_23_analysis_ready_half_diam.gpkg'
CONTRACTS_JSON = WORK / RUN_ID / 'flight_contracts.json'
TRANSLATION_REGISTRY = REPO / 'provenance/drone-calibration-translation/2026-11-19/aop-translation-audit/aop_derived_micasense_to_landsat_fractional_reflectance_registry.json'
TARGETS = ('Landsat_8_OLI', 'Landsat_9_OLI-2')
DELIVERY_NAMING = 'micasense'
OUTPUT_SCALE_FACTOR = 1000.0  # Stored target values = physical fractional reflectance ×1000.
GOCMD = 'gocmd'
PILOT_PACKAGE = 'AOP-Ruby-08-14-23-ExportPackage'  # Must exactly match an eligible inventory package.
MIN_FREE_DISK_GIB = 50  # Operational floor only; budget the full campaign separately.
REMOTE_READS = False
STAGE_PILOT_INPUTS = False
RUN_PILOT = False
RUN_ALL = False
UPLOAD = False


In [ ]:
import inspect
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logging.getLogger('spectralbridge').setLevel(logging.INFO)
import spectralbridge.pipelines.drone as drone
from spectralbridge import inspect_drone_collection
sys.path.insert(0, str(REPO / 'scripts'))
from drone_cloud_workflow import JsonGocmdBackend, select_inventory
from drone_vm_campaign import run_vm_campaign, upload_verified_campaign
from spectralbridge.bulk.provenance import write_json_atomic

assert Path(drone.__file__).resolve().is_relative_to(REPO / 'src'), 'Restart kernel after editable install.'
assert {'brightness_calibration', 'camera_geometry', 'processing_mask', 'translation_output_scale_factor'}.issubset(inspect.signature(drone.run_drone_pipeline).parameters), 'VM checkout is missing the campaign integration.'
assert MANIFEST.is_file() and TRANSLATION_REGISTRY.is_file()
assert POLYGONS is None or POLYGONS.is_file()
WORK.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage(WORK).free >= MIN_FREE_DISK_GIB * 2**30, 'Insufficient free persistent disk for configured floor.'
print('Package:', drone.__file__)
print('Revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Working changes:', subprocess.check_output(['git', 'status', '--short'], cwd=REPO, text=True))
print('Free disk GiB:', round(shutil.disk_usage(WORK).free / 2**30, 1))


## 2. Inventory all requested years before large transfers

This checks the existing acquisition manifest; no hard-coded flight count substitutes for completeness. Camera coverage for all 2024 flights must be established from their original-year collections. Missing packages, camera/height evidence, invalid calibration or unsupported model fits remain visible; the campaign cannot be uploaded as complete while a flight is blocked.


In [ ]:
assert REMOTE_READS, 'Review configuration, enable REMOTE_READS and rerun configuration.'
backend = JsonGocmdBackend(GOCMD)
backend.ensure_available()
inventory = inspect_drone_collection(
    RASTER_SOURCES, years=YEARS, manifest=MANIFEST, backend=backend,
    inventory_output_dir=WORK / RUN_ID / 'inventory',
)
print(json.dumps(inventory.year_summaries(), indent=2, default=str))
print('Excluded:', [(p.package_name, p.reason) for p in inventory.excluded_packages])
assert inventory.discovery_complete and inventory.eligible_packages, 'Resolve manifest/year coverage first.'


## 3. Complete the per-flight execution contracts

`flight_contracts.json` is keyed by each exact remote raster ExportPackage path. Each entry contains:

- `source_sha256`: original H5 bytes to which the reviewed calibration applies; `wavelengths_nm`: exact retained-band wavelengths/order; `source_radiometry`: verified source decoding record, distinct from the new brightness gains.
- `brightness_json`: VM path to the incoming reviewed calibration declaration. Schema: `schema_version=1`, `status="REVIEWED"`, `calibration_id`, `flight_id`, `source_sha256`, `provenance`, `equation="calibrated_reflectance = gain * decoded_reflectance + offset"`, `bands=[{"wavelength_nm": ..., "gain": ..., "offset": ...}, ...]`. Offsets are fractional reflectance; confirm direction and units with the provider. Adapt their JSON once, preserve it, and do not invent coefficients.
- `camera_collection`: original-year package collection. `camera_geometry`: `assignment_policy`, `height_datum_provenance`, `max_view_zenith_deg`, `provenance`, and optional evidence-backed `crs_override`/`crs_override_provenance`. Policy `reviewed_index` requires an exact-grid raster of original zero-based camera-CSV row numbers (`-1` unknown). Alternatively `nearest_camera_scenario` requires `assumption_acknowledgement` and retains an UNKNOWN-contributors warning. No camera scalar or nadir fallback is supplied.
- `auxiliary_sources`: remote paths for `surface` and, for reviewed indices, `camera_index`. Optional `processing_mask` names a categorical mask; its separate `processing_mask` policy declares `keep_value`, `allowed_values` and `provenance`. The surface must use the source grid and the documented camera height datum. No silent spatial resampling or vertical offset is inferred.
- `pipeline`: explicit `apply_topo`, `apply_brdf`, `extraction_mode` (`polygon`, `full`, or `qa_only`), `parquet_chunk_size`, and `drone_manifest_timezone`. Use reviewed photo/manifest timing; do not assume UTC solely to unblock correction. A 10% shade screen is not made a universal default. The estimator/reference/QA remain the existing public pipeline contracts, not a silent promotion of a frozen experimental adapter.

The starter below writes a **separate template**, never overwrites a reviewed contract, and deliberately leaves unknown scientific values empty. Inspect actual source identities/camera files using the readiness notebook. Actual source bytes, retained TIFF/H5 equality and wavelength identities are rechecked during execution. Goldhill's documented CRS discrepancy needs independent provenance; it is never automatically relabelled.


In [ ]:
template = {}
for package in inventory.eligible_packages:
    template[package.remote_package_path] = {
        'source_sha256': None, 'wavelengths_nm': None, 'source_radiometry': None,
        'brightness_json': str(WORK / RUN_ID / 'calibration' / f'{package.flight_stem}.json'),
        'camera_collection': f'{CAMERA_SOURCES[package.year]}/{package.package_name}',
        'camera_geometry': {
            'assignment_policy': 'reviewed_index', 'height_datum_provenance': None,
            'max_view_zenith_deg': None, 'provenance': None,
        },
        'auxiliary_sources': {'surface': None, 'camera_index': None},
        'pipeline': {
            'apply_topo': None, 'apply_brdf': None, 'extraction_mode': 'polygon',
            'parquet_chunk_size': 2048, 'drone_manifest_timezone': None,
        },
    }
template_path = WORK / RUN_ID / 'flight_contracts.template.json'
if not template_path.exists():
    write_json_atomic(template_path, template)
print('Template:', template_path)
print('Reviewed execution contracts:', CONTRACTS_JSON)


### Recover the pilot's source identity before the coefficients arrive

If the provider JSON does not include original file identities, enable `STAGE_PILOT_INPUTS` to download the pilot's unchanged source bundle on the VM. This writes exact SHA-256 and spectral metadata for its contract without applying any calibration. Source decoding remains a separate reviewed declaration; an unknown record is printed as unsupported, never inferred from a 10k filename.


In [ ]:
if STAGE_PILOT_INPUTS:
    import h5py
    from drone_cloud_workflow import download_sources
    from spectralbridge.pipelines.drone import _find_drone_reflectance_dataset
    from spectralbridge.drone_campaign_inputs import file_sha256
    from spectralbridge.radiometry import dataset_radiometry
    package = select_inventory(inventory, PILOT_PACKAGE).eligible_packages[0]
    staging = WORK / 'staged_sources' / f'{package.year}_{package.package_name}'
    download_sources(package, staging, backend)
    original = staging / package.h5_name
    with h5py.File(original, 'r') as h:
        ds = _find_drone_reflectance_dataset(h)
        source_identity = {
            'flight_id': package.flight_stem,
            'source_sha256': file_sha256(original),
            'wavelengths_nm': ds.parent['Metadata/Spectral_Data/Wavelength'][:].tolist(),
            'source_radiometry': dataset_radiometry(ds).record(),
        }
    write_json_atomic(WORK / RUN_ID / 'pilot_source_identity.json', source_identity)
    print(json.dumps(source_identity, indent=2))
else:
    print('Pilot source staging disabled.')


## 4. One-flight pilot

Set `RUN_PILOT=True` after completing the selected flight's contract and incoming brightness JSON, then rerun configuration and this cell. Contract/translation checks run before large raster transfers. The original H5 and aligned TIFF/terrain are staged one package at a time; original-year camera/surface/index/mask assets are downloaded explicitly. The ten-band TIFF recipe must exactly match the original H5. It replaces inherited scalar views with declared per-pixel camera geometry on a derived working H5.

Decoding and brightness calibration compose once. Masks apply to fitting and application with recorded exclusions. Outputs use existing naming helpers; OLI/OLI2 ENVI stored values are fractional reflectance ×1000, with the inverse affine and divisor in the header. Native products, Parquet and QA retain physical-unit provenance. These synthetic-response translations are Landsat-like products, not independently observed satellite reflectance.

Camera/mask construction and new-mode correction application are row-bounded. The existing producer still loads one flight's reflectance cube in memory: allocate RAM for that cube plus correction/QA working arrays. Persistent disk holds the entire output campaign plus staged sources. Readback later needs free space for one largest output object. No giant state/pixel cache is created by this orchestration.


In [ ]:
contracts = json.loads(CONTRACTS_JSON.read_text()) if CONTRACTS_JSON.is_file() else {}
if RUN_PILOT:
    pilot_inventory = select_inventory(inventory, PILOT_PACKAGE)
    pilot = run_vm_campaign(
        pilot_inventory, contracts, work=WORK, run_id=RUN_ID, backend=backend,
        manifest=MANIFEST, polygons=POLYGONS, translation_registry=TRANSLATION_REGISTRY,
        targets=TARGETS, naming_style=DELIVERY_NAMING, output_scale_factor=OUTPUT_SCALE_FACTOR,
        scope='pilot',
    )
    print(json.dumps(pilot, indent=2))
    assert pilot['status'] == 'complete', 'Inspect blocked/failed stages before proceeding.'
else:
    print('Pilot disabled; complete its contract and enable RUN_PILOT.')


## 5. Review the pilot, then run the full population

Inspect the saved camera/source/calibration provenance, applied correction flags, every band/model's numerical support, mask coverage, corrected/translated images and PNG/JSON QA. Model QA is not independent reflectance accuracy. Newly calibrated flights may fail BRDF even if the earlier three-site examples passed; failures are not filled with identity models or unrequested alternative branches.

The runner revalidates the pilot against current calibration contents, registry, source/geometry contracts, delivery choices and code hashes; changed inputs require a new pilot. Keep `RUN_ID` while resuming. The runner fingerprints calibration contents, registry, policy and implementation: changed inputs get a fresh output root so older calibrated products survive. Unchanged inputs reuse their working/export/correction/translation checkpoints. Failed flights remain recorded while other flights continue. The full campaign is kept separately from the explicitly scoped pilot; do not present a one-flight upload as campaign completion.


In [ ]:
PILOT_QA_REVIEWED = False  # Set True only after reviewing the actual pilot artifacts.
if RUN_ALL:
    assert 'pilot' in globals() and pilot['status'] == 'complete', 'Complete the one-flight pilot first.'
    assert PILOT_QA_REVIEWED, 'Review the actual pilot QA and provenance before full execution.'
    campaign = run_vm_campaign(
        inventory, contracts, work=WORK, run_id=RUN_ID, backend=backend,
        manifest=MANIFEST, polygons=POLYGONS, translation_registry=TRANSLATION_REGISTRY,
        targets=TARGETS, naming_style=DELIVERY_NAMING, output_scale_factor=OUTPUT_SCALE_FACTOR,
        scope='campaign', pilot=pilot,
    )
    print(json.dumps(campaign, indent=2))
    assert campaign['status'] == 'complete', 'Resolve every blocked/failed manifest flight before upload.'
else:
    print('Full campaign disabled; finish and review the pilot first.')


## 6. Upload processed products with full readback, then publish completion

Set `UPLOAD=True` after reviewing the complete campaign. The destination is a new identity-named collection beneath `spectralbridge_processed`; changed payloads receive distinct collections. Uploads include processed flight outputs, native backing H5, translated products, extraction tables, QA, provider/execution calibration JSON, camera CSVs and source/auxiliary transfer provenance. Original raster staging is excluded.

Each transfer uses GoCommands checksum verification. Every uploaded payload file is then downloaded to one temporary file on the persistent VM disk and checked against local SHA-256. Only after all readbacks pass is `completion/campaign_complete.json` uploaded and itself read back. An interrupted/failed verification leaves no new completed-campaign receipt. Repeated transfers resume using existing diff/checksum helpers. This verifies bytes/completeness, not independent scientific accuracy.


In [ ]:
if UPLOAD:
    assert 'campaign' in globals() and campaign['status'] == 'complete', 'Complete and review the full population first.'
    upload = upload_verified_campaign(campaign, backend=backend, remote_results_parent=REMOTE_RESULTS_PARENT)
    write_json_atomic(Path(campaign['run_root']) / 'upload_receipt.json', upload)
    print(json.dumps(upload, indent=2))
else:
    print('Uploads disabled. Processed outputs remain on the persistent VM disk.')
